# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we enumerate all `RecordSet` entities (`@type`: `cr:RecordSet`) along with their available fields (`cr:field`). All references are by their `@id` field for clarity and reproducibility.

In [ ]:
# Find and print all record sets and their fields by @id
from mlcroissant._src.structure import RecordSet, Field

record_sets = []
print('RecordSets in the dataset (by `@id`):')
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id} ; Name: {rs.name}")
    record_sets.append(rs.id)
    if rs.fields:
        print("    Fields (by @id):")
        for field in rs.fields:
            print(f"      - {field.id} : {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we extract all record sets into separate DataFrames, referencing each key entity by their `@id`.

In [ ]:
# List all record set IDs found previously (copy from overview cell if needed)
record_set_ids = record_sets  # Should be a list of @id strings from above
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")

# Inspect first record set in detail (if at least one exists)
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in DataFrame for {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You should select a numeric field and a group-by field from the record set columns (reference by `@id`). Example field selection is shown below; update these variables as appropriate based on your dataset.

In [ ]:
# ---- EDIT THESE VARIABLES TO MATCH YOUR DATASET STRUCTURE ----
# Select a record set to analyze
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id:
    df = dataframes[record_set_id]
    print(f"Exploring record set: {record_set_id}")

    # List available columns for selection
    print("Available columns:", df.columns.tolist())

    # Suggest likely numeric fields from dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
    print(f"Numeric candidate fields: {numeric_fields}")
    print(f"Group-by candidate fields: {group_fields}")

    # ---- SELECT FIELD IDS for analysis ----
    # For illustration, pick the first numeric field (edit for your case):
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)  # Example threshold: upper quartile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (top quartile):")
        display(filtered_df.head())

        # Normalization
        z = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = z
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optional grouping
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found for this record set.")
else:
    print("No record sets found in this dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we provide simple histograms and box plots for the selected numeric field, and bar plots for a grouped summary if grouping was performed above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_fields:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(5,4))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    if group_fields:
        plt.figure(figsize=(12,5))
        top_cats = filtered_df[group_field_id].value_counts().index[:10]  # Only show top 10 groups
        ax = sns.barplot(
            x=group_field_id,
            y=numeric_field_id,
            data=filtered_df[filtered_df[group_field_id].isin(top_cats)],
        )
        plt.title(f"{numeric_field_id} by {group_field_id} (top 10 categories)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset of mass spectrometry analysis using the `mlcroissant` library:
- We reviewed all record sets and fields by `@id` as defined in the Croissant metadata.
- We demonstrated how to extract all records into DataFrames for analysis.
- We showed basic EDA and field normalization, referencing fields strictly by their Croissant `@id`.
- We provided simple visualizations for numeric fields.

This notebook can serve as a starter for deeper, domain-specific analysis of this and similar datasets structured with Croissant/FAIR standards.